# Environmental hazards and safety incidents
## Mexicali Urban Liveability Index — `WP05_hazards_and_incidents`

**Lead:** TBC
**Indicators assigned:** 8
**Schema version:** 1.0.0

Flooding exposure (including main roads), fire incidence, hazardous waste, crime, and road traffic crashes and fatalities.

Incident registries are counts of reported events; normalise by population or exposure and be explicit about reporting bias and the risk of penalising well-surveilled areas.

> New to this project? Work through
> [`00_overview_and_schema.ipynb`](00_overview_and_schema.ipynb)
> first — it carries one indicator end to end. Then read
> [`docs/analyst_guide.md`](../docs/analyst_guide.md).

## How to work through this notebook

For each indicator assigned to you, in this order:

1. **Read the brief.** It reproduces everything the team already
   recorded in the workbook — the draft rationale, the article the
   indicator was adapted from, candidate data sources, and the open
   questions colleagues raised. Do not retype any of it; it is
   already in your metadata stub.
2. **Write the causal pathway sentence** (guide §2.1) and find
   **independent health evidence** for it (§2.2). Do this *before*
   looking for data. Fill in `meta['rationale']`.
3. **Find and document the data** (§3): citation, URL, date
   retrieved, licence, and whether it reaches Condesa.
4. **Compute** at the finest scale your data genuinely support.
   Produce a `DataFrame` with `geo_id` and `value`.
5. **Harmonise** with `uli.harmonise(...)`, label with
   `uli.label(...)`, and **deliver** with
   `uli.write_indicator(...)`.
6. **Look at the map.** Most errors are obvious in ten seconds and
   invisible in a table.

`uli.write_indicator` validates first and refuses to publish a
failing deliverable. While you are still iterating, pass
`allow_failure=True` to write a draft anyway.

Full guidance: [`docs/analyst_guide.md`](../docs/analyst_guide.md).
Schema: [`schema/ULI_output_schema.md`](../schema/ULI_output_schema.md).

## Framing the indicator against health evidence

Every indicator must be justified by evidence of a **meaningful
health or wellbeing benefit**, independent of the liveability
article it was adapted from. Those articles establish that an
indicator is used; they rarely establish that it matters.

Complete this sentence before you compute anything:

> *[what I measure]* changes *[a mechanism]*, which changes *[a
> behaviour or exposure]*, which affects *[a health outcome]*.

For most indicators in this project the behaviour is **walking for
transport**, **walking or recreation in public space**, or
**social contact** — and the exposure is **heat**, **air
pollution** or **injury risk**. Say which, using the vocabulary in
`uli.vocab.HEALTH_PATHWAYS`.

Prefer meta-analyses and systematic reviews, then reputable
guidance (WHO, UN-Habitat, PAHO, Secretaría de Salud), then cohort
studies and natural experiments. Record the **effect size with its
uncertainty**.

**If the evidence supports a different threshold from the one the
workbook proposes, use the evidence-based threshold** and say so in
`threshold_justification`. That is explicitly what the project
wants.

**Mexicali is arid and extremely hot.** Most of this literature
comes from temperate cities. Where the transfer is doubtful — for
example, distance-based walkability thresholds in a city where
summer maxima exceed 45 °C and shade rather than distance is the
binding constraint — record it in `rationale.arid_context`. That is
a contribution, not a caveat.

## When several workbook rows are really one indicator

The workbook harvested indicators article by article, so a single
construct sometimes appears as several rows seen through different
lenses or over different time periods. Air quality is the clearest
case:

| Row | What it is | Lens | Time basis |
|---|---|---|---|
| #292 Air quality | the index value itself | `quality` | `annual_mean` |
| #8 Good air quality | that value against a standard | `quality` | `threshold_share` |
| #293 Days with good air quality | how often the standard is met | `quantity` | `threshold_compliance_days` |
| #173 Days PM2.5 over WHO | the same, for one pollutant | `quantity` | `threshold_exceedance_days` |

These are not four indicators — they are one construct measured
four ways, and computing them separately would mean four
inconsistent methods and four sets of data documentation.

Deliver them as a **measure family**: give every measure the same
`measure_family` slug, and distinguish them with `temporal_basis`
(see `uli.vocab.TEMPORAL_BASES`) and `threshold`. They can still
live under separate workbook ids — the family slug is what tells
the index step, and Reimagina Urbana, that they belong together.

```python
for meta in (meta_292, meta_8, meta_293, meta_173):
    for measure in meta['measures']:
        measure['measure_family'] = 'air_quality'
    meta['data_sources'] = SHARED_SOURCES   # one method, one source
```

The same pattern applies to mean summer temperature versus days
above a comfort threshold (WP02), and to flood extent versus annual
average days of flooding (WP05).

## Condesa coverage is a requirement, not a nicety

The Condesa new development in south-east Mexicali is a project
focus area, and it defeats the usual assumptions:

- about **20%** of it falls outside the previously configured
  study region boundary;
- only **44%** of its area is covered by census manzana polygons,
  so a **manzana-native calculation reaches 33 of the 40
  fraccionamientos, while a `grid_100m`-native one reaches all
  40**;
- it is platted and roaded (43 km of street network in OpenStreetMap
  across 27 of the 40 fraccionamientos) but essentially unbuilt —
  **zero destinations**, and satellite-derived population products
  see almost nobody there.

**One thing is your decision: the native scale.** If your data
allow it, compute on the 100 m grid. That is the difference between
reaching all of Condesa and quietly missing a fifth of it.

Everything else is handled downstream. Population denominators,
the 2030 occupancy scenario and population-weighted exposure
statistics are a reporting-step concern (`uli.exposure`), decided
once for the whole project rather than by each analyst. Urban
fabric and exposure measures — land cover, air quality, heat,
hazards, street infrastructure — are properties of *place*, and
should be computed as such; who lives there is applied later.

Two things to record, though:

- `data_sources[].condesa_coverage` — whether your **source**
  reaches Condesa. Satellite imagery and OSM generally do; a 2020
  census variable or a household survey generally does not.
- `method.condesa_treatment` — what you did about it. Where a
  source does not reach Condesa, mark those rows `no_data` rather
  than omitting them.

The validator treats poor Condesa coverage as an **error**.

---
## Setup

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import uli

# Identify yourself once; it is copied into every deliverable.
ANALYST = {
    'name': 'TODO: your name',
    'email': None,
    'institution': None,
}

print(f'ULI schema version {uli.SCHEMA_VERSION}')
print(f'Reference geographies available: {uli.geography.available()}')

        WORK_PACKAGE = 'WP05_hazards_and_incidents'
        NOTEBOOK = 'notebooks/05_hazards_and_incidents.ipynb'

---
## Your indicators (8)

Each has a brief reproducing what the workbook records,
then three working cells: documentation, calculation,
delivery.

### 296 — Crime?

`crime` · *Safety · Personal Safety · Crime · Crime?*

- **Lenses to deliver:** accessibility
- **Draft rationale (rewrite this):** Personal safety and the reduction of urban risk through protective infrastructure are identified as fundamental requirements for city liveability and resident well-being.
- **Adapted from:** Mercer, 2017, 'Quality of Living Survey'; Economist Intelligence Unit (EIU), 2012, 'Quality of Life Index'
- **Effect reported there:** Liveability score: weighted indicator, no individual effect size reported
- **Methods used in the literature:** ['Hierarchical Modelling using Analytic Hierarchy Process (AHP)']
- **Open questions raised:** CH: Only useful if it can be acquired with meaningful variation

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_296) at any time to list what is
# still outstanding.
meta_296 = uli.metadata_stub(296, analyst=ANALYST)

# meta_296['rationale']['statement'] = """..."""
# meta_296['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_296['rationale']['arid_context'] = '...'
# meta_296['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_296['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_296)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_296 = 'grid_100m'
METHOD_296 = 'population_weighted_mean'

native_296 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_296 = uli.harmonise(
    native_296,
    native_scale=NATIVE_SCALE_296,
    method=METHOD_296,
)
results_296 = uli.label(
    harmonised_296,
    meta_296,
    measure_id='crime__accessibility',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_296, meta_296))
# uli.write_indicator(results_296, meta_296)

### 325 — Fire incidence rate by population

`fire_incidence_rate_by_population` · *Safety/Ambient Environment · Risks · Fire · Fire incidence rate by population*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** Fire safety, measured by the occurrence rate in the population, is a critical component of urban security, essential for protecting lives and property.
- **Adapted from:** Arpan & Joy, 2018, 'Livability assessment within a metropolis based on the impact of integrated urban geographic factors (IUGFs) on clustering urban centers of Kolkata'; Zhan et al., 2018, 'Assessment and determinants of satisfaction with urban livability in China'
- **Effect reported there:** No health outcome assessed.
- **Methods used in the literature:** ['Not specified']
- **Team notes:** ER: In Mexicali the Risk Atlas states that the most relevant fires are those related to industry, however, there is no data available for these.
- **Feasibility flag:** to be determined

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_325) at any time to list what is
# still outstanding.
meta_325 = uli.metadata_stub(325, analyst=ANALYST)

# meta_325['rationale']['statement'] = """..."""
# meta_325['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_325['rationale']['arid_context'] = '...'
# meta_325['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_325['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_325)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_325 = 'grid_100m'
METHOD_325 = 'population_weighted_mean'

native_325 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_325 = uli.harmonise(
    native_325,
    native_scale=NATIVE_SCALE_325,
    method=METHOD_325,
)
results_325 = uli.label(
    harmonised_325,
    meta_325,
    measure_id='fire_incidence_rate_by_population__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_325, meta_325))
# uli.write_indicator(results_325, meta_325)

### 71 — Flooding

`flooding` · *Ambient Environment · Risks · Flooding · Flooding*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** Mitigating the exposure and impact of urban flooding is essential for maintaining residents' health and safety, particularly for those in informal and vulnerable housing.
- **Adapted from:** No external citations provided for this indicator.
- **Effect reported there:** No health outcome assessed.
- **Methods used in the literature:** ['N/A']
- **Candidate data sources:** Risk Atlas layers for Mexicali (https://www.mexicali.gob.mx/sitioimip/geovisor/geovisor/?geovisor_id=2&access_token=)
- **Feasibility flag:** yes, data believed available

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_71) at any time to list what is
# still outstanding.
meta_71 = uli.metadata_stub(71, analyst=ANALYST)

# meta_71['rationale']['statement'] = """..."""
# meta_71['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_71['rationale']['arid_context'] = '...'
# meta_71['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_71['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_71)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_71 = 'grid_100m'
METHOD_71 = 'population_weighted_mean'

native_71 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_71 = uli.harmonise(
    native_71,
    native_scale=NATIVE_SCALE_71,
    method=METHOD_71,
)
results_71 = uli.label(
    harmonised_71,
    meta_71,
    measure_id='flooding__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_71, meta_71))
# uli.write_indicator(results_71, meta_71)

### 72 — Flooding - main roads

`flooding_main_roads` · *Ambient Environment · Risks · Flooding · Flooding - main roads*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** The number of flood-prone locations on main roads is an indicator of infrastructure vulnerability that directly affects environmental safety and health risk management.
- **Adapted from:** Parise, 2018, 'A brief review of global climate change and the public health consequences'
- **Effect reported there:** Association reported; no effect size provided.
- **Methods used in the literature:** ['Identification of measurable spatial indicators, how they can me calculated and monitored']
- **Candidate data sources:** Risk Atlas layers for Mexicali (https://www.mexicali.gob.mx/sitioimip/geovisor/geovisor/?geovisor_id=2&access_token=)
- **Feasibility flag:** yes, data believed available

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_72) at any time to list what is
# still outstanding.
meta_72 = uli.metadata_stub(72, analyst=ANALYST)

# meta_72['rationale']['statement'] = """..."""
# meta_72['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_72['rationale']['arid_context'] = '...'
# meta_72['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_72['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_72)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_72 = 'grid_100m'
METHOD_72 = 'population_weighted_mean'

native_72 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_72 = uli.harmonise(
    native_72,
    native_scale=NATIVE_SCALE_72,
    method=METHOD_72,
)
results_72 = uli.label(
    harmonised_72,
    meta_72,
    measure_id='flooding_main_roads__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_72, meta_72))
# uli.write_indicator(results_72, meta_72)

### 326 — Flooding - annual average days for main roads

`flooding_annual_average_days_for_main_roads` · *Ambient Environment · Risks · Flooding · Flooding - annual average days for main roads*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** The duration of flooding in main road areas contributes to environmental risks and disruption of essential services, which can impact public health and safety.
- **Adapted from:** Parise, 2018, 'A brief review of global climate change and the public health consequences'
- **Effect reported there:** Association reported; no effect size provided.
- **Methods used in the literature:** ['Identification of measurable spatial indicators, how they can me calculated and monitored']
- **Feasibility flag:** to be determined

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_326) at any time to list what is
# still outstanding.
meta_326 = uli.metadata_stub(326, analyst=ANALYST)

# meta_326['rationale']['statement'] = """..."""
# meta_326['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_326['rationale']['arid_context'] = '...'
# meta_326['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_326['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_326)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_326 = 'grid_100m'
METHOD_326 = 'population_weighted_mean'

native_326 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_326 = uli.harmonise(
    native_326,
    native_scale=NATIVE_SCALE_326,
    method=METHOD_326,
)
results_326 = uli.label(
    harmonised_326,
    meta_326,
    measure_id='flooding_annual_average_days_for_main_roads__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_326, meta_326))
# uli.write_indicator(results_326, meta_326)

### 235 — Road accident locations

`road_accident_locations` · *Safety · Road Safety · Pedestrian Infrastructure · Road accident locations*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** Monitoring road accident locations supports the development of safer transport infrastructure to reduce injuries and fatalities from vehicular traffic.
- **Adapted from:** Lowe et al., 2015, 'Planning healthy, liveable and sustainable cities: how can indicators inform policy?'
- **Effect reported there:** Association reported; no effect size provided.
- **Methods used in the literature:** ['Identification of measurable spatial indicators, how they can me calculated and monitored']
- **Candidate data sources:** INEGI Accidentes de Tránsito: https://www.inegi.org.mx/programas/accidentes/#documentacion

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_235) at any time to list what is
# still outstanding.
meta_235 = uli.metadata_stub(235, analyst=ANALYST)

# meta_235['rationale']['statement'] = """..."""
# meta_235['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_235['rationale']['arid_context'] = '...'
# meta_235['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_235['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_235)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_235 = 'grid_100m'
METHOD_235 = 'population_weighted_mean'

native_235 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_235 = uli.harmonise(
    native_235,
    native_scale=NATIVE_SCALE_235,
    method=METHOD_235,
)
results_235 = uli.label(
    harmonised_235,
    meta_235,
    measure_id='road_accident_locations__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_235, meta_235))
# uli.write_indicator(results_235, meta_235)

### 334 — Traffic accident fatalities

`traffic_accident_fatalities` · *Safety · Road Safety · Pedestrian Infrastructure · Traffic accident fatalities*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** Traffic accident fatalities are a primary indicator of road safety; reduced mortality is a fundamental requirement for a secure and livable city environment.
- **Adapted from:** Kourtit et al., 2021, 'Safe cities in the new urban world: A comparative cluster dynamics analysis through machine learning'; Arpan & Joy, 2018, 'Livability assessment within a metropolis based on the impact of integrated urban geographic factors (IUGFs) on clustering urban centers of Kolkata'
- **Effect reported there:** No health outcome assessed.
- **Methods used in the literature:** ['Not specified']
- **Candidate data sources:** INEGI Accidentes de Tránsito: https://www.inegi.org.mx/programas/accidentes/#documentacion

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_334) at any time to list what is
# still outstanding.
meta_334 = uli.metadata_stub(334, analyst=ANALYST)

# meta_334['rationale']['statement'] = """..."""
# meta_334['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_334['rationale']['arid_context'] = '...'
# meta_334['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_334['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_334)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_334 = 'grid_100m'
METHOD_334 = 'population_weighted_mean'

native_334 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_334 = uli.harmonise(
    native_334,
    native_scale=NATIVE_SCALE_334,
    method=METHOD_334,
)
results_334 = uli.label(
    harmonised_334,
    meta_334,
    measure_id='traffic_accident_fatalities__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_334, meta_334))
# uli.write_indicator(results_334, meta_334)

### 341 — Annual hazardous waste (kg)

`annual_hazardous_waste_kg` · *Safety/Ambient Environment · Risks · Waste Management · Annual hazardous waste (kg)*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** Monitoring hazardous waste volumes is essential for preventing the accumulation of toxic substances in the environment that can cause severe illness or injury.
- **Adapted from:** Giles-Corti et al., 2019, 'Achieving the SDGs: evaluating indicators to be used to benchmark and monitor progress towards creating healthy and sustainable cities'
- **Effect reported there:** Association reported; no effect size provided.
- **Methods used in the literature:** ['Identification of measurable spatial indicators, how they can me calculated and monitored']
- **Team notes:** ER: We can obtain data for some industries from the Taking Stock website
- **Feasibility flag:** yes, data believed available

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_341) at any time to list what is
# still outstanding.
meta_341 = uli.metadata_stub(341, analyst=ANALYST)

# meta_341['rationale']['statement'] = """..."""
# meta_341['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_341['rationale']['arid_context'] = '...'
# meta_341['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_341['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_341)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_341 = 'grid_100m'
METHOD_341 = 'population_weighted_mean'

native_341 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_341 = uli.harmonise(
    native_341,
    native_scale=NATIVE_SCALE_341,
    method=METHOD_341,
)
results_341 = uli.label(
    harmonised_341,
    meta_341,
    measure_id='annual_hazardous_waste_kg__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_341, meta_341))
# uli.write_indicator(results_341, meta_341)

---
## Check what this work package has delivered

In [ ]:
delivered, catalogue = uli.collect()
if len(catalogue):
    display(catalogue)
    print(delivered.groupby(['indicator_code', 'geo_level']).size())
else:
    print('Nothing delivered yet.')

In [ ]:
# Sanity-check a delivered measure on a map before you call it done.
# MEASURE = 'your_indicator_code__quantity'
# LEVEL = 'manzana'
# units = uli.geography.load(LEVEL).merge(
#     delivered.query('measure_id == @MEASURE and geo_level == @LEVEL'),
#     on='geo_id', how='left')
# ax = units.plot(column='value', legend=True, figsize=(11, 8),
#                 missing_kwds={'color': 'lightgrey'})
# condesa = uli.geography.load('condesa_fraccionamiento')
# condesa.boundary.plot(ax=ax, color='red', linewidth=1)
# ax.set_title(MEASURE)
# ax.set_axis_off()